# Theory vs manual vs CNC-gantry axial profile — 2026-07-23

Compares three views of the Bessel-beam axial intensity profile after axicon #3:

1. **QDHT model** — now imported from `simulator/qdht_axicon.py` (extracted
   2026-07-23 from the copies in the two `*_2026_07_13` notebooks; "third time"
   rule). Same physics, same parameters as before.
2. **Manual XY scans, 2026-07-13** (`data/manual_scan-2026-07-13_*`) — peak
   count rate per z, method identical to `exposure_vs_z_analysis_2026_07_13.ipynb`.
3. **CNC gantry auto scans, 2026-07-22/23** (`profiler/data/auto_scan-*`) —
   per-slice peak count rate from `frames.jsonl` manifests (per-frame `Max`,
   `Exposure_us`), i.e. **full sensor resolution** — deliberately *not* from the
   8×-downsampled composites, which mean-pool the ~58 µm core away.

Conventions and decisions:

- x-axis is distance after axicon3 in cm: gantry `TableY` (anchored via
  `MeasuredSensorY_mm` per placement) ≡ manual `SensorZ_cm`. Both anchors are
  tape measurements, so ±0.5 cm on both.
- **Absolute counts/µs, no cross-normalization** between manual and gantry
  (Gain 0 dB, Mono8, gamma off on both days). Any offset in the 70–90 cm
  overlap region is shown as-is — it measures day-to-day power/alignment drift
  plus placement-anchor error. The model alone is scaled (single factor) to the
  manual peak.
- **Peak-pixel caveat**: outside the Bessel window the brightest pixel is a
  *converging/diverging annulus fringe*, not the on-axis core (same effect as
  the lobe-FWHM caveat in the jul13 notebook). The model is therefore plotted
  two ways: on-axis intensity, and **max-over-r intensity at each z** — the
  latter is the apples-to-apples comparator for a peak-pixel metric, and the
  only meaningful one below ~110 cm where all the gantry data lives.
- **Flagging rule**: a gantry slice is excluded (hollow marker) if any lit
  frame saturated — at slice hand-offs the find-beam exposure escalation (×8)
  saturates the frames that actually cover the beam, so the surviving
  unsaturated `Max` is off-beam and biased low. Affects the last slice of
  several runs (e.g. y=18, 54, 71.5 cm and all three y=89.5 cm slices).
- Background: off-axis background frames average ≲0.002 counts/px at these
  exposures — negligible, but subtracted anyway.

TODO(leigh): confirm the laser wavelength (650 nm assumed, as before), and the
1/e² *diameter* reading of GaussianBeamWaist (see jul13 notebook header).

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")
from simulator.qdht_axicon import simulate_recorded_optic

GANTRY_ROOT = Path("../profiler/data")
MANUAL_ROOT = Path("../data")
WAVELENGTH_NM = 650          # TODO: confirm laser wavelength
Z_ERR_MANUAL_CM = 0.5        # tape-measure z reading
Y_ERR_GANTRY_CM = 0.5        # MeasuredSensorY anchor is also tape-measured;
                             # gantry steps within a placement are ~10 um
QDHT_ATTRIBUTION = "Quasi-Discrete Hankel Transform model (Yu et al. 1998)"

NOTEBOOK_STEM = "theory_vs_manual_vs_gantry_2026_07_23"
IMAGES_DIR = Path("images") / NOTEBOOK_STEM
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(description):
    path = IMAGES_DIR / f"{description}.png"
    plt.gcf().savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")

## Ensure every auto-scan slice is composited (ds 2)

The on-axis extraction and the slice gallery below read `composite.npy` from
every slice, and everything in this notebook assumes the **downsample-2**
(6.9 µm/px) composites — coarser ones pool away the core and fringes. This
step walks all `auto_scan-*` runs and builds any composite that is missing or
was made at a different downsample, using the same `composite_slice` call as
`dataset composite` (so new scans dropped into `profiler/data/` are picked up
by just re-running the notebook). Slices already at ds 2 are skipped, so the
pass is cheap (~2 s per rebuilt slice, no-op otherwise). The manual jul13
runs are untouched — their stitched composites come from the legacy
registration-based stitcher and are already full-resolution.

In [ ]:
sys.path.append("../profiler")
from composite import CompositeConfig, composite_slice, CompositeError

COMPOSITE_DOWNSAMPLE = 2
composite_config = CompositeConfig(
    PixelSize_um=3.45, Downsample=COMPOSITE_DOWNSAMPLE,
    FlipX=True, FlipZ=True, Transpose=False,       # camera 180° vs machine axes
    SubtractBackground=True, IncludeDarkFrames=False, Colormap="inferno",
)

built, skipped, failures = 0, 0, []
for run_dir in sorted(GANTRY_ROOT.glob("auto_scan-*")):
    for slice_dir in sorted(run_dir.glob("y*cm")):
        if not (slice_dir / "frames.jsonl").exists():
            continue          # aborted slice (no manifest) — nothing to build
        meta_p = slice_dir / "composite_meta.json"
        if meta_p.exists():
            try:
                if json.loads(meta_p.read_text())["Downsample"] == COMPOSITE_DOWNSAMPLE:
                    skipped += 1
                    continue
                print(f"rebuilding {slice_dir.relative_to(GANTRY_ROOT)}: "
                      f"existing composite at wrong downsample")
            except (json.JSONDecodeError, KeyError) as e:
                print(f"rebuilding {slice_dir.relative_to(GANTRY_ROOT)}: "
                      f"unreadable composite_meta.json ({e})")
        try:
            composite_slice(slice_dir, composite_config, output_stem="composite")
            built += 1
        except (CompositeError, OSError, ValueError) as e:
            failures.append(slice_dir)
            print(f"FAILED {slice_dir.relative_to(GANTRY_ROOT)}: "
                  f"{type(e).__name__}: {e} — this slice will be MISSING from "
                  f"the on-axis extraction and the slice gallery")

print(f"composites: {built} built, {skipped} already at ds{COMPOSITE_DOWNSAMPLE}, "
      f"{len(failures)} failed")
if failures:
    print("WARNING: proceeding without the failed slices listed above.")

## Gantry reduction — per-slice peak rate from `frames.jsonl`

No pixel data is touched except the (tiny) background frames; the manifests
already carry per-frame `Max`, `SaturatedPixelCount` and `Exposure_us`.
A cached copy of this reduction as of 2026-07-23 lives in
`reductions/axial_reduction_2026_07_23.json` (note: taken while run
`15-04-24` was still growing — re-running this cell picks up whatever is on
disk now).

In [ ]:
gantry = []
for run_dir in sorted(GANTRY_ROOT.glob("auto_scan-*")):
    setup_path = run_dir / "auto_scan_setup.json"
    if not setup_path.exists():
        print(f"skipping {run_dir.name}: no auto_scan_setup.json")
        continue
    setup = json.loads(setup_path.read_text())
    for slice_dir in sorted(run_dir.glob("y*cm")):
        manifest = slice_dir / "frames.jsonl"
        if not manifest.exists():
            print(f"skipping {slice_dir.relative_to(GANTRY_ROOT)}: no frames.jsonl")
            continue
        lit, bg = [], []
        with manifest.open() as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                name = Path(rec["Path"]).name
                if "background" in name:
                    bg.append(rec)
                elif "-dark" not in name:
                    lit.append(rec)
        if not lit:
            continue
        bg_means = [float(np.load(slice_dir / Path(r["Path"]).name).mean())
                    for r in bg if (slice_dir / Path(r["Path"]).name).exists()]
        bg_mean = float(np.mean(bg_means)) if bg_means else 0.0

        best, n_sat = None, 0
        for rec in lit:
            if rec.get("SaturatedPixelCount", 0) > 0:
                n_sat += 1
                continue
            rate = (rec["Max"] - bg_mean) / rec["Extra"]["Exposure_us"]
            if best is None or rate > best["rate"]:
                best = {"rate": rate, "max": rec["Max"],
                        "T_us": rec["Extra"]["Exposure_us"]}
        if best is None:
            print(f"dropping {slice_dir.relative_to(GANTRY_ROOT)}: "
                  f"all {n_sat} lit frames saturated")
            continue
        gantry.append({
            "run": run_dir.name,
            "placement": setup.get("PlacementID"),
            "beam_y_cm": lit[0]["Extra"]["BeamY_mm"] / 10.0,
            "rate": best["rate"],
            "n_lit": len(lit),
            "flagged": n_sat > 0,   # saturated find-beam frames -> peak biased low
        })

runs = sorted({g["run"] for g in gantry})
n_flag = sum(g["flagged"] for g in gantry)
print(f"{len(gantry)} slices from {len(runs)} runs; {n_flag} flagged (hollow markers)")

## Manual jul13 reduction — same method as the jul13 notebook

Peak from `composite.npy` where stitched (feather-averaged — same count units
as raw frames), else from the single frame; divided by the calibrated
exposure. Background 0 pending the dark-frame runs (same placeholder as the
jul13 notebook).

In [ ]:
manual = []
for run_dir in sorted(MANUAL_ROOT.glob("manual_scan-2026-07-13_*")):
    setup_path = run_dir / "sweep_setup.json"
    if not setup_path.exists():
        continue
    setup = json.loads(setup_path.read_text())
    T = setup["CalibratedExposure_us"]
    frames = sorted(p for p in run_dir.glob("*.npy")
                    if not p.name.startswith("composite"))
    composite = run_dir / "composite.npy"
    if composite.exists():
        peak = float(np.nanmax(np.asarray(np.load(composite, mmap_mode="r"))))
    elif len(frames) == 1:
        peak = float(np.load(frames[0]).max())
    else:
        print(f"skipping {run_dir.name}: multi-frame but no composite")
        continue
    manual.append({"z_cm": setup["SensorZ_cm"], "rate": peak / T})

man_z = np.array([m["z_cm"] for m in manual])
man_rate = np.array([m["rate"] for m in manual])
print(f"{len(manual)} manual runs, z {man_z.min():g}-{man_z.max():g} cm")

# Optic configuration: recorded identically in both datasets - verify, then use.
OPTIC = json.loads((sorted(GANTRY_ROOT.glob("auto_scan-*"))[-1]
                    / "auto_scan_setup.json").read_text())["Metadata"]["OpticConfiguration"]
manual_optic = json.loads(
    (sorted(MANUAL_ROOT.glob("manual_scan-2026-07-13_*"))[0]
     / "sweep_setup.json").read_text())["OpticConfiguration"]
assert all(abs(OPTIC[k] - manual_optic[k]) < 1e-9 for k in OPTIC), (
    f"optic configs differ between datasets: {OPTIC} vs {manual_optic}")
OPTIC_TITLE = (
    f"axicon #1,2 alpha={OPTIC['Axicon1_deg']:g} deg, "
    f"L12 separation={OPTIC['L12_mm']:.1f} mm, "
    f"axicon #3 alpha={OPTIC['Axicon3_deg']:g} deg, "
    f"L23 separation={OPTIC['L23_mm']:.1f} mm\n"
    f"input Gaussian waist {OPTIC['GaussianBeamWaist_mm']:.1f} mm, "
    f"wavelength {WAVELENGTH_NM:g} nm"
)
print("OpticConfiguration:", OPTIC)

## QDHT model — on-axis *and* brightest-feature curves

`simulate_recorded_optic` reproduces the jul13 notebook's model run
(GaussianBeamWaist read as 1/e² diameter). The **max-over-r** curve is the
comparator for a peak-pixel metric: outside the window it follows the annulus
fringe — exactly what the camera's brightest pixel reports there.

In [ ]:
pair, model = simulate_recorded_optic(OPTIC, N=4096, Nz2=40, Nz3=400)
z_model_cm = model["z3"] * 100.0

r_eval = np.linspace(0.0, model["q"].R, 3000)
M_eval = model["q"].eval_matrix(r_eval)
I_max = np.empty(len(model["z3"]))
step = 64
for i0 in range(0, len(model["z3"]), step):
    rows = np.abs(model["Y3"][i0:i0 + step] @ M_eval) ** 2
    I_max[i0:i0 + step] = rows.max(axis=1)

# One global scale: model -> manual peak (at the window peak the axis IS the
# brightest pixel, so the same factor serves both model curves).
scale = man_rate.max() / I_max.max()
print(f"model peak at z = {z_model_cm[np.argmax(model['onax'])]:.1f} cm, "
      f"scale to manual peak = {scale:.3g}")

## Axial profile: model vs manual vs gantry

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 6))
ax.plot(z_model_cm, I_max * scale, "-", color="tab:orange", lw=1.6,
        label="QDHT model, brightest feature at each z\n"
              "(not necessarily on-axis)",
        zorder=2)
ax.plot(z_model_cm, model["onax"] * scale, "--", color="tab:orange", lw=1.1,
        alpha=0.55, label="QDHT model, on-axis only", zorder=1)
ax.errorbar(man_z, man_rate, xerr=Z_ERR_MANUAL_CM, fmt="o", ms=6, capsize=2,
            color="tab:blue", label="manual XY scans 2026-07-13 (peak rate)",
            zorder=4)

cmap = plt.get_cmap("viridis")
run_color = {r: cmap(0.15 + 0.7 * i / max(len(runs) - 1, 1))
             for i, r in enumerate(runs)}
seen = set()
for g in gantry:
    c = run_color[g["run"]]
    label = None
    if g["run"] not in seen and not g["flagged"]:
        seen.add(g["run"])
        label = (f"gantry {g['run'].split('auto_scan-')[1][:16]} "
                 f"({g['placement']})")
    ax.errorbar(g["beam_y_cm"], g["rate"], xerr=Y_ERR_GANTRY_CM, fmt="s",
                ms=5, color=c, markerfacecolor="none" if g["flagged"] else c,
                label=label, zorder=3, alpha=0.9)
ax.errorbar([], [], fmt="s", ms=5, color="0.4", markerfacecolor="none",
            label="gantry, excluded (saturated find-beam frames)",)

ax.set_yscale("log")
ax.set_xlim(0, 280)
ax.set_ylim(1e-4, 2.0)
ax.set_xlabel("distance after axicon3 (cm)   [gantry TableY = manual sensor z]")
ax.set_ylabel("peak pixel rate (counts/µs)")
ax.set_title(
    "Peak pixel rate vs distance after axicon #3 — QDHT model vs manual "
    "(jul13) vs CNC gantry (jul22–23).\n"
    "Absolute counts/µs; no cross-normalization between datasets. "
    f"{QDHT_ATTRIBUTION}\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.005, 0.5))
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("axial_profile_theory_manual_gantry")
plt.show()

## Cross-checks: overlap region and gantry repeatability

Left: the 60–100 cm region where both datasets exist. Right: y positions
covered by two different gantry runs (jul23-003 re-scanned 46–54 cm after a
soft-limit widening; jul23-005 re-scanned 84.5–89.5 cm) — the spread is the
short-term repeatability of the whole pipeline (positioning + calibration +
laser).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

sel = (man_z >= 60) & (man_z <= 100)
ax1.plot(z_model_cm, I_max * scale, "-", color="tab:orange", lw=1.6,
         label="QDHT model, brightest feature")
ax1.plot(z_model_cm, model["onax"] * scale, "--", color="tab:orange", lw=1.1,
         alpha=0.55, label="QDHT model, on-axis")
ax1.errorbar(man_z[sel], man_rate[sel], xerr=Z_ERR_MANUAL_CM, fmt="o", ms=6,
             capsize=2, color="tab:blue", label="manual jul13")
for g in gantry:
    if 60 <= g["beam_y_cm"] <= 100 and not g["flagged"]:
        ax1.errorbar(g["beam_y_cm"], g["rate"], xerr=Y_ERR_GANTRY_CM,
                     fmt="s", ms=5, color=run_color[g["run"]])
ax1.set_xlim(58, 100)
ax1.set_yscale("log")
ax1.set_ylim(1e-4, 3e-2)
ax1.set_xlabel("distance after axicon3 (cm)")
ax1.set_ylabel("peak pixel rate (counts/µs)")
ax1.set_title("Overlap region: gantry (jul23) vs manual (jul13)\n"
              "offset = day-to-day power/alignment drift + placement anchor",
              fontsize=9)
ax1.legend(fontsize=8)
ax1.minorticks_on(); ax1.grid(True, which="both", alpha=0.3)

by_y = defaultdict(list)
for g in gantry:
    if not g["flagged"]:
        by_y[round(g["beam_y_cm"], 2)].append(g["rate"])
dups = {y: r for y, r in by_y.items() if len(r) > 1}
for y, rates in sorted(dups.items()):
    mid = np.mean(rates)
    ax2.plot([y] * len(rates), 100 * (np.array(rates) / mid - 1), "o", ms=6,
             color="0.3")
ax2.axhline(0, color="0.6", lw=0.8)
ax2.set_xlabel("distance after axicon3 (cm)")
ax2.set_ylabel("deviation from per-y mean (%)")
ax2.set_title("Gantry repeatability: same y, different runs", fontsize=9)
ax2.minorticks_on(); ax2.grid(True, which="both", alpha=0.3)

fig.suptitle("Gantry vs manual cross-checks. " + OPTIC_TITLE, fontsize=9)
plt.tight_layout()
save_fig("overlap_and_repeatability")
plt.show()

# quantify: gantry/manual ratio in the overlap, and repeatability spreads
ov_y = [g["beam_y_cm"] for g in gantry
        if not g["flagged"] and g["beam_y_cm"] >= 69]
ov_r = [g["rate"] for g in gantry
        if not g["flagged"] and g["beam_y_cm"] >= 69]
order = np.argsort(man_z)
ratios = np.array(ov_r) / np.interp(ov_y, man_z[order], man_rate[order])
print(f"overlap (>=69 cm): gantry/manual ratio median {np.median(ratios):.2f} "
      f"(n={len(ratios)})")
spreads = [100 * (max(r) / min(r) - 1) for r in dups.values()]
print(f"repeatability across duplicate y: median spread "
      f"{np.median(spreads):.1f}%, worst {max(spreads):.1f}% "
      f"(n={len(spreads)} duplicated positions)")

## Reading the comparison (as of the 2026-07-23 reduction)

- **Rising edge, 7–90 cm (gantry)**: the measured brightest-pixel rate rises
  smoothly ×4.5 across the region and is continuous across placements
  (jul22-001 → jul23-005) — the TableY anchoring works. It sits a factor of a
  few *above* the model's brightest-feature curve at low z. Candidate
  explanations, in order: the model's annulus fringe contrast is
  resolution-limited (N=4096 ⇒ ~5 pts/fringe), the ±0.5° axicon tolerances,
  and the overall anchor of the model scale to the manual (drifted) peak.
- **Overlap 70–90 cm**: gantry runs ~1.6× above the jul13 manual points —
  consistent with day-to-day laser power/alignment drift plus the tape-measure
  anchor; to be split apart, this needs the same-day A/B (manual placement +
  auto scan back-to-back) or a power-meter reading logged per run.
- **Repeatability**: duplicate-y slices from different runs agree to a median
  ~5% (worst ~13%) — that bounds short-term drift + recalibration noise, and is
  the natural error bar for the gantry points.
- **Window region**: only the manual data covers it so far; the known
  peak-position (155 vs 163 cm) and width mismatches from the jul13 notebook
  stand unchanged. The gantry hasn't reached the window yet — extending the
  y-coverage past ~110 cm (new placement further downstream) is the obvious
  next scan.
- **On-axis profile: now done below** (2026-07-23, after recompositing all
  slices at `--downsample 2`) — see the "On-axis extraction" section at the
  end of this notebook.

## On-axis extraction (added 2026-07-23)

A true on-axis profile below the window, from the `--downsample 2` composites
(all jul22–23 slices were recomposited at 6.9 µm/px on 2026-07-23):

1. **Find the axis**: weighted Taubin circle fit to the bright annulus in each
   slice composite (pixels above max(3, 0.25·peak) counts, intensity-weighted).
   Validation: the fitted ring radius shrinks 5.8 → 2.3 mm from y = 7 → 89.5 cm,
   tracking the geometric prediction r(y) ≈ R_a − θ₃·y (≈0.4 mm inside it —
   intensity-weighted ring-width bias + R_a tolerance), and the center is
   continuous across placements.
2. **Read the axis**: mean composite value in a **0.15 mm radius aperture** at
   the fitted center, ÷ slice exposure. Lucky geometry: the 7.1×5.3 mm frame
   footprint means every ring center is covered by ≥1 *lit* frame — no
   include-dark composites needed.
3. **Compare like with like**: the model curve is the QDHT on-axis intensity
   **averaged over the same 0.15 mm aperture** (area-weighted disk mean). This
   matters twice: it smooths the model's deep radial interference nulls
   (40–55 cm), and inside the window it dilutes the 58 µm core the same way
   the measured aperture would.

Flags: slices with a bad circle fit (|r−pred| > 1.5 mm, e.g. the 23-frame
find-beam slice y26) are excluded; ring-saturated slices are kept but hollow —
saturation sits on the ring, the center pixels are unsaturated and the slice
exposure is uniform (asserted from the manifest).

Caveats: composite background subtraction clips at 0 (small positive bias on
few-count signals); aperture pixels are correlated (2×2 pooling + overlap
averaging) so the error bars use n_eff = n/4; the aperture mean would dilute a
real 58 µm axial spike ~10× — a spike-resolved (center-pixel) comparison
against the exact r=0 model curve is a possible refinement.

In [ ]:
# --- circle-fit axis finder + on-axis aperture extraction --------------------
APERTURE_MM = 0.15
R_A_MM = model["R_a"] * 1e3          # model annulus radius at axicon3 [mm]
TH3 = model["th3"]                   # cone half-angle after axicon3


def taubin_circle_fit(x, z, w):
    """Weighted Taubin algebraic circle fit. Returns (xc, zc, r)."""
    W = w / w.sum()
    xm, zm = (W * x).sum(), (W * z).sum()
    u, v = x - xm, z - zm
    Suu = (W * u * u).sum(); Svv = (W * v * v).sum(); Suv = (W * u * v).sum()
    Suuu = (W * u**3).sum(); Svvv = (W * v**3).sum()
    Suvv = (W * u * v * v).sum(); Svuu = (W * v * u * u).sum()
    A = np.array([[Suu, Suv], [Suv, Svv]])
    b = 0.5 * np.array([Suuu + Suvv, Svvv + Svuu])
    uc, vc = np.linalg.solve(A, b)
    return xm + uc, zm + vc, float(np.sqrt(uc**2 + vc**2 + Suu + Svv))


onaxis = []
for run_dir in sorted(GANTRY_ROOT.glob("auto_scan-*")):
    for slice_dir in sorted(run_dir.glob("y*cm")):
        comp_p = slice_dir / "composite.npy"
        meta_p = slice_dir / "composite_meta.json"
        if not (slice_dir / "frames.jsonl").exists() or not comp_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        if meta["Downsample"] != 2:
            print(f"skipping {slice_dir.name}: composite at ds={meta['Downsample']}, expected 2")
            continue
        comp = np.load(comp_p)
        ext, s = meta["Extent_mm"], meta["mm_per_px"]

        # ring fit on bright pixels (machine-mm coordinates)
        thr = max(3.0, 0.25 * comp.max())
        rows, cols = np.nonzero(comp > thr)
        vals = comp[rows, cols].astype(np.float64)
        x_mm = ext["XMin"] + (cols + 0.5) * s
        z_mm = ext["ZMax"] - (rows + 0.5) * s
        xc, zc, r_fit = taubin_circle_fit(x_mm, z_mm, vals)
        pred_r = R_A_MM - TH3 * meta["BeamY_mm"]

        # slice exposure + ring-saturation flag from the manifest
        exposures, n_sat = set(), 0
        with (slice_dir / "frames.jsonl").open() as fh:
            for line in fh:
                if not line.strip():
                    continue
                rec = json.loads(line)
                name = Path(rec["Path"]).name
                if "background" in name or "-dark" in name:
                    continue
                exposures.add(round(rec["Extra"]["Exposure_us"], 1))
                n_sat += rec.get("SaturatedPixelCount", 0) > 0
        if len(exposures) != 1:
            print(f"skipping {slice_dir.name}: mixed exposures {sorted(exposures)}")
            continue
        T = exposures.pop()

        # aperture mean about the fitted center
        col_c = (xc - ext["XMin"]) / s - 0.5
        row_c = (ext["ZMax"] - zc) / s - 0.5
        rr, cc = np.mgrid[0:comp.shape[0], 0:comp.shape[1]]
        ap = (np.hypot(cc - col_c, rr - row_c) * s) <= APERTURE_MM
        vals_ap = comp[ap]
        onaxis.append({
            "slice": f"{run_dir.name}/{slice_dir.name}",
            "beam_y_cm": meta["BeamY_mm"] / 10.0,
            "xc_mm": round(xc, 3), "zc_mm": round(zc, 3),
            "r_mm": round(r_fit, 3), "pred_r_mm": round(pred_r, 3),
            "onax_counts": float(vals_ap.mean()),
            "onax_std": float(vals_ap.std()),
            "n_ap_px": int(ap.sum()), "exposure_us": T,
            "onax_rate": float(vals_ap.mean()) / T,
            "sat_flag": bool(n_sat > 0),
            "fit_flag": bool(abs(r_fit - pred_r) > 1.5),
        })

Path("reductions").mkdir(exist_ok=True)
Path("reductions/onaxis_reduction_2026_07_23.json").write_text(
    json.dumps(onaxis, indent=1))
n_fit = sum(o["fit_flag"] for o in onaxis)
print(f"{len(onaxis)} slices extracted; {n_fit} excluded for bad circle fit; "
      f"aperture signals {min(o['onax_counts'] for o in onaxis):.1f}"
      f"-{max(o['onax_counts'] for o in onaxis):.1f} counts")

In [ ]:
# --- model on-axis averaged over the same aperture + comparison figure -------
r_ap = np.linspace(0.0, APERTURE_MM * 1e-3, 60)
M_ap = model["q"].eval_matrix(r_ap)
w_ap = r_ap.copy(); w_ap[0] = r_ap[1] / 4          # area weights (r dr)
model_onax_ap = np.empty(len(model["z3"]))
for i0 in range(0, len(model["z3"]), 64):
    I = np.abs(model["Y3"][i0:i0 + 64] @ M_ap) ** 2
    model_onax_ap[i0:i0 + 64] = (I * w_ap).sum(axis=1) / w_ap.sum()

good = [o for o in onaxis if not o["fit_flag"]]
bad = [o for o in onaxis if o["fit_flag"]]
oy = np.array([o["beam_y_cm"] for o in good])
orate = np.array([o["onax_rate"] for o in good])
oerr = np.array([o["onax_std"] / np.sqrt(o["n_ap_px"] / 4) / o["exposure_us"]
                 for o in good])
osat = np.array([o["sat_flag"] for o in good])

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(z_model_cm, model_onax_ap * scale, "--", color="tab:orange", lw=1.4,
        label="QDHT model, on-axis averaged over 0.15 mm aperture\n"
              "(scaled to manual window peak)")
ax.plot(z_model_cm, model["onax"] * scale, ":", color="tab:orange", lw=0.9,
        alpha=0.45, label="QDHT model, exact r=0")
ax.plot(z_model_cm, I_max * scale, "-", color="tab:orange", lw=1.0,
        alpha=0.4, label="QDHT model, brightest feature (context)")
ax.errorbar(oy[~osat], orate[~osat], yerr=oerr[~osat], fmt="o", ms=5,
            capsize=2, color="tab:green",
            label="gantry on-axis (circle-fit center, 0.15 mm aperture)")
ax.errorbar(oy[osat], orate[osat], yerr=oerr[osat], fmt="o", ms=5, capsize=2,
            color="tab:green", markerfacecolor="none",
            label="gantry on-axis, ring-saturated slice (center unsaturated)")
for o in bad:
    ax.plot(o["beam_y_cm"], o["onax_rate"], "x", color="0.5", ms=7)
ax.plot([], [], "x", color="0.5", label="excluded: bad circle fit")
ax.errorbar(man_z, man_rate, xerr=Z_ERR_MANUAL_CM, fmt="o", ms=5, capsize=2,
            alpha=0.5, color="tab:blue", label="manual jul13 peak rate (context)")

ax.set_yscale("log")
ax.set_xlim(0, 200)
ax.set_ylim(1e-5, 2)
ax.set_xlabel("distance after axicon3 (cm)")
ax.set_ylabel("count rate (counts/µs)")
ax.set_title(
    "Measured on-axis intensity below the window — circle-fit extraction vs "
    "QDHT on-axis model.\nAbsolute counts/µs; model scaled once to the manual "
    "window peak.\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.legend(fontsize=8, loc="upper left")
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("onaxis_vs_model")
plt.show()

m_interp = np.interp(oy, z_model_cm, model_onax_ap * scale)
for lo, hi in [(0, 30), (30, 60), (60, 92)]:
    msk = (oy >= lo) & (oy < hi)
    if msk.any():
        print(f"y {lo}-{hi} cm: measured/model(on-axis, aperture-avg) median "
              f"{np.median(orate[msk] / m_interp[msk]):7.2f}  (n={msk.sum()})")

### Reading the on-axis result

- **The measurement worked**: a smooth, placement-continuous on-axis curve
  from 3.5×10⁻⁵ to 2×10⁻⁴ counts/µs across y = 7–89.5 cm, with aperture
  signals of 1.5–38 counts — above the quantization floor everywhere.
- **The pre-window axis is far brighter than the ideal model**: measured/model
  ≈ ×600 at y < 30 cm, ≈ ×25 at 30–60 cm, converging to ≈ ×0.6 by 60–90 cm.
  The ideal-axicon model says the axis should be almost perfectly dark this
  far below the window; the measured interior glow is the signature of
  **axicon imperfections — apex rounding above all** (a rounded apex acts as
  a weak lens throwing light on-axis early), plus scatter from the three
  axicon surfaces. This is a real, known effect in axicon systems, not noise.
- Near the window edge (60–90 cm) the measured curve rises *slower* than the
  model's steep leading edge — consistent with the window-narrowing already
  seen in the manual data.
- Follow-ups this suggests: model apex rounding explicitly (replace the ideal
  conical phase with a rounded-apex profile in `simulate_third_axicon` — the
  QDHT machinery needs no other change); log laser power per run to separate
  drift from physics; and extend gantry coverage past y ≈ 110 cm so the
  on-axis extraction runs into the window where the model is trustworthy.

## Peak-normalized view (linear scale, jul13 plot style)

Same data as the axial-profile figure, in the normalization of the original
jul13 plot: **all measured points ÷ the manual window peak** (one constant —
this is the absolute comparison re-expressed in units of the window peak, so
the gantry/manual day-to-day offset is still visible), model ÷ its own peak.
On a linear axis the gantry rising edge lives below 0.012, so the inset
magnifies y < 100 cm ×50.

In [ ]:
P = man_rate.max()                      # measured window peak (jul13)
gy = np.array([g["beam_y_cm"] for g in gantry])
gr = np.array([g["rate"] for g in gantry]) / P
gflag = np.array([g["flagged"] for g in gantry])
model_norm = I_max / I_max.max()
meas_norm = man_rate / P

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(z_model_cm, model_norm, "-", color="tab:orange", lw=1.6,
        label="QDHT model (brightest feature, normalized to its peak)")
ax.errorbar(man_z, meas_norm, xerr=Z_ERR_MANUAL_CM, fmt="o", ms=6, capsize=2,
            color="tab:blue", label="manual jul13 peak rate ÷ window peak")
ax.errorbar(gy[~gflag], gr[~gflag], xerr=Y_ERR_GANTRY_CM, fmt="s", ms=5,
            color="tab:green", label="gantry jul22–23 peak rate ÷ window peak")
ax.errorbar(gy[gflag], gr[gflag], xerr=Y_ERR_GANTRY_CM, fmt="s", ms=5,
            color="tab:green", markerfacecolor="none",
            label="gantry, flagged (saturated find-beam frames)")
ax.set_xlim(0, 280)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel("distance after axicon3 (cm)")
ax.set_ylabel("intensity, normalized to the measured window peak")
ax.set_title(
    "Peak-normalized comparison (linear scale). Measured data ÷ manual jul13 "
    "window peak; model ÷ its own peak.\n" + OPTIC_TITLE, fontsize=10)
ax.legend(fontsize=8, loc="upper right")
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)

# axin = ax.inset_axes([0.055, 0.35, 0.36, 0.52])
# axin.plot(z_model_cm, model_norm, "-", color="tab:orange", lw=1.4)
# axin.errorbar(gy[~gflag], gr[~gflag], xerr=Y_ERR_GANTRY_CM, fmt="s", ms=4,
#               color="tab:green")
# axin.errorbar(gy[gflag], gr[gflag], xerr=Y_ERR_GANTRY_CM, fmt="s", ms=4,
#               color="tab:green", markerfacecolor="none")
# sel = man_z <= 100
# axin.errorbar(man_z[sel], meas_norm[sel], xerr=Z_ERR_MANUAL_CM, fmt="o",
#               ms=4, color="tab:blue")
# axin.set_xlim(0, 100)
# axin.set_ylim(0, 0.022)
# axin.set_title("y < 100 cm (×50 zoom)", fontsize=8)
# axin.tick_params(labelsize=7)
# axin.grid(True, alpha=0.3)
# ax.indicate_inset_zoom(axin, edgecolor="0.5")

plt.tight_layout()
save_fig("peak_normalized_comparison")
plt.show()

## Gantry X-Z slice gallery

The jul13-style slice grid, for the CNC data: every jul22–23 slice's ds-2
composite, cropped ±7 mm about its **fitted ring center** (from the on-axis
section — a centroid would be biased by partial coverage), normalized
per-panel, sqrt scale. Gray = outside adaptive-raster coverage. Duplicated y
values (re-scanned runs) appear as adjacent panels. SAT/FIT! flags as in the
on-axis section — the SAT panels are visibly blown out, and the y26 FIT!
panel shows the find-beam sweep contamination.

In [ ]:
GALLERY_HALF_MM = 7.0
panels = []
for o in sorted(onaxis, key=lambda o: (o["beam_y_cm"], o["slice"])):
    slice_dir = GANTRY_ROOT / o["slice"]
    comp = np.load(slice_dir / "composite.npy")
    meta = json.loads((slice_dir / "composite_meta.json").read_text())
    ext, s = meta["Extent_mm"], meta["mm_per_px"]
    half_px = int(round(GALLERY_HALF_MM / s))
    col_c = int(round((o["xc_mm"] - ext["XMin"]) / s))
    row_c = int(round((ext["ZMax"] - o["zc_mm"]) / s))
    out = np.full((2 * half_px, 2 * half_px), np.nan, dtype=np.float32)
    r0, c0 = row_c - half_px, col_c - half_px
    rs0, cs0 = max(r0, 0), max(c0, 0)
    rs1 = min(r0 + 2 * half_px, comp.shape[0])
    cs1 = min(c0 + 2 * half_px, comp.shape[1])
    out[rs0 - r0:rs1 - r0, cs0 - c0:cs1 - c0] = comp[rs0:rs1, cs0:cs1]
    out[out == 0.0] = np.nan               # uncovered canvas -> gray
    step = max(1, (2 * half_px) // 400)
    run = o["slice"].split("/")[0]
    panels.append((o, run.split("auto_scan-")[1].replace("_", " ")[:16],
                   out[::step, ::step]))

n_cols = 6
n_rows = int(np.ceil(len(panels) / n_cols))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.15")
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.55 * n_cols, 2.75 * n_rows))
for ax in axes.ravel():
    ax.set_axis_off()
for ax, (o, stamp, panel) in zip(axes.ravel(), panels):
    ax.set_axis_on()
    im = ax.imshow(np.sqrt(np.clip(panel / np.nanmax(panel), 0, 1)),
                   cmap=cmap, vmin=0, vmax=1,
                   extent=[-GALLERY_HALF_MM, GALLERY_HALF_MM,
                           -GALLERY_HALF_MM, GALLERY_HALF_MM],
                   interpolation="nearest")
    flag = " SAT" if o["sat_flag"] else (" FIT!" if o["fit_flag"] else "")
    ax.set_title(f"y = {o['beam_y_cm']:g} cm{flag}\n{stamp}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(
    "CNC gantry X-Z slices vs distance after axicon #3 (jul22-23), "
    f"normalized per-panel, sqrt scale, ±{GALLERY_HALF_MM:g} mm about fitted "
    "ring center.\n" + OPTIC_TITLE + "\nGray = outside adaptive-raster "
    "coverage. SAT = ring-saturated slice; FIT! = unreliable circle fit.",
    fontsize=12, y=0.998)
fig.colorbar(im, ax=axes, label="sqrt(normalized intensity)",
             fraction=0.015, pad=0.01)
save_fig("gantry_xz_slices_grid")
plt.show()

## Near-axis uniformity — is the bright region big enough for a 300 µm Sr cloud?

Requirement under test: the atom cloud (≈300 µm diameter → r = 150 µm) should
sit in a *uniformly bright* near-axis region.

Method, per slice: azimuthal radial profile about the beam axis (1-px ≈ 6.9 µm
bins from the ds-2 composite; center = circle-fit ring center, refined to the
core centroid when a bright core is present, i.e. in/near the window). Each
bin records the azimuthal mean and the p10–p90 spread over angle, so
ellipticity, centering error, and stray fringes count against uniformity.
Metrics:

- **r_u(±tol)** — largest radius such that every bin inside it keeps its
  p10–p90 envelope within ±tol of the central intensity I₀ (mean over
  r ≤ 21 µm). Reported for ±10/25/50%. Values at 21 µm mean "≤ 21 µm"
  (metric floor: inside the I₀ definition disk).
- **C150** — contrast (I_max−I_min)/I_mean over r ≤ 150 µm: the intensity
  variation across the cloud footprint. ±10% uniformity ⇔ C150 ≤ 0.2.

Same metrics computed from the QDHT model I(r, z). Ring-saturated and
bad-fit slices are excluded. Note the pre-window measured p10/p90 envelope is
quantization-limited (few-count signals), so treat measured r_u(±10%) there
as a lower bound; C150 and the window region are unaffected (30+ counts).

In [ ]:
# --- azimuthal radial profiles + uniformity metrics per slice ----------------
CLOUD_R_UM = 150.0
UNIF_TOLS = (0.10, 0.25, 0.50)
R_MAX_MM = 2.2

def radial_uniformity(slice_dir):
    comp = np.load(slice_dir / "composite.npy")
    meta = json.loads((slice_dir / "composite_meta.json").read_text())
    if meta["Downsample"] != 2:
        raise ValueError(f"composite at ds={meta['Downsample']}, expected 2")
    ext, s = meta["Extent_mm"], meta["mm_per_px"]

    # center: ring fit, refined to core centroid when a bright core exists
    thr = max(3.0, 0.25 * comp.max())
    rows, cols = np.nonzero(comp > thr)
    vals = comp[rows, cols].astype(np.float64)
    x_mm = ext["XMin"] + (cols + 0.5) * s
    z_mm = ext["ZMax"] - (rows + 0.5) * s
    xc, zc, r_fit = taubin_circle_fit(x_mm, z_mm, vals)
    col_c = (xc - ext["XMin"]) / s - 0.5
    row_c = (ext["ZMax"] - zc) / s - 0.5
    rr, cc = np.mgrid[0:comp.shape[0], 0:comp.shape[1]]
    near = (np.hypot(cc - col_c, rr - row_c) * s) <= 0.6
    center = "ring-fit"
    if near.any() and comp[near].max() >= 0.5 * comp.max():
        pr, pc = np.unravel_index(np.argmax(np.where(near, comp, -1)), comp.shape)
        w = comp[max(pr-4, 0):pr+5, max(pc-4, 0):pc+5].astype(np.float64)
        yy, xx = np.mgrid[max(pr-4, 0):pr+5, max(pc-4, 0):pc+5]
        row_c = float((w * yy).sum() / w.sum())
        col_c = float((w * xx).sum() / w.sum())
        center = "core-centroid"

    # slice exposure + saturation flag
    exposures, n_sat = set(), 0
    with (slice_dir / "frames.jsonl").open() as fh:
        for line in fh:
            if not line.strip():
                continue
            rec = json.loads(line)
            name = Path(rec["Path"]).name
            if "background" in name or "-dark" in name:
                continue
            exposures.add(round(rec["Extra"]["Exposure_us"], 1))
            n_sat += rec.get("SaturatedPixelCount", 0) > 0
    if len(exposures) != 1:
        raise ValueError(f"{len(exposures)} distinct exposures")
    T = exposures.pop()

    # radial binning, 1-px bins
    half = int(R_MAX_MM / s) + 2
    rs0, cs0 = max(int(row_c) - half, 0), max(int(col_c) - half, 0)
    rs1 = min(int(row_c) + half, comp.shape[0])
    cs1 = min(int(col_c) + half, comp.shape[1])
    crop = comp[rs0:rs1, cs0:cs1].astype(np.float64)
    yy, xx = np.mgrid[rs0:rs1, cs0:cs1]
    rbin = np.hypot(xx - col_c, yy - row_c).astype(int)
    nbins = int(R_MAX_MM / s)
    ok = rbin < nbins
    rb, v = rbin[ok], crop[ok]
    order = np.argsort(rb, kind="stable")
    rb, v = rb[order], v[order]
    edges = np.searchsorted(rb, np.arange(nbins + 1))
    mean = np.full(nbins, np.nan)
    p10 = np.full(nbins, np.nan)
    p90 = np.full(nbins, np.nan)
    for b in range(nbins):
        vb = v[edges[b]:edges[b + 1]]
        if len(vb) < 0.3 * 2 * np.pi * (b + 0.5):   # partial coverage -> NaN
            continue
        mean[b] = vb.mean()
        p10[b] = np.percentile(vb, 10)
        p90[b] = np.percentile(vb, 90)
    mean, p10, p90 = mean / T, p10 / T, p90 / T
    r_um = (np.arange(nbins) + 0.5) * s * 1000.0

    I0 = np.nanmean(mean[r_um <= 21])
    r_u = {}
    for tol in UNIF_TOLS:
        viol = np.where(~np.isnan(mean)
                        & ((p10 < (1 - tol) * I0) | (p90 > (1 + tol) * I0)))[0]
        viol = viol[r_um[viol] > 21]
        r_u[tol] = float(r_um[viol[0]] - 3.45) if len(viol) else float(r_um[-1])
    disk = (r_um <= CLOUD_R_UM) & ~np.isnan(mean)
    C150 = float((np.nanmax(p90[disk]) - np.nanmin(p10[disk]))
                 / np.nanmean(mean[disk]))
    pred_r = (model["R_a"] * 1e3) - model["th3"] * meta["BeamY_mm"]
    return dict(beam_y_cm=meta["BeamY_mm"] / 10.0, I0_rate=I0, r_u=r_u,
                C150=C150, center=center, r_um=r_um, mean=mean, p10=p10,
                p90=p90, sat_flag=n_sat > 0,
                fit_flag=(abs(r_fit - pred_r) > 1.5 and center == "ring-fit"))


radial = []
for run_dir in sorted(GANTRY_ROOT.glob("auto_scan-*")):
    for slice_dir in sorted(run_dir.glob("y*cm")):
        if not (slice_dir / "composite_meta.json").exists():
            continue
        try:
            rec = radial_uniformity(slice_dir)
        except (ValueError, OSError, np.linalg.LinAlgError) as e:
            print(f"skipping {slice_dir.relative_to(GANTRY_ROOT)}: "
                  f"{type(e).__name__}: {e}")
            continue
        rec["slice"] = str(slice_dir.relative_to(GANTRY_ROOT))
        radial.append(rec)
radial.sort(key=lambda r: (r["beam_y_cm"], r["slice"]))
radial_good = [r for r in radial if not r["sat_flag"] and not r["fit_flag"]]
print(f"{len(radial)} slices profiled, {len(radial_good)} clean")

In [ ]:
# --- model metrics + uniform-radius figure -----------------------------------
r_m = np.linspace(0.0, R_MAX_MM * 1e-3, 640)
M_r = model["q"].eval_matrix(r_m)
r_um_model = r_m * 1e6
core_m = r_um_model <= 21
disk_m = r_um_model <= CLOUD_R_UM
model_ru = {t: np.full(len(z_model_cm), np.nan) for t in UNIF_TOLS}
model_c150 = np.full(len(z_model_cm), np.nan)
for i0 in range(0, len(z_model_cm), 64):
    I = np.abs(model["Y3"][i0:i0 + 64] @ M_r) ** 2
    for j, row in enumerate(I):
        I0 = row[core_m].mean()
        for t in UNIF_TOLS:
            viol = np.where((np.abs(row - I0) > t * I0) & (r_um_model > 21))[0]
            model_ru[t][i0 + j] = (r_um_model[viol[0]] if len(viol)
                                   else r_um_model[-1])
        model_c150[i0 + j] = ((row[disk_m].max() - row[disk_m].min())
                              / row[disk_m].mean())

fig, ax = plt.subplots(figsize=(11, 6))
tol_color = {0.10: "tab:red", 0.25: "tab:purple", 0.50: "tab:brown"}
for t in UNIF_TOLS:
    ax.plot(z_model_cm, model_ru[t], "-", color=tol_color[t], lw=1.3,
            alpha=0.7, label=f"model, ±{int(t*100)}%")
    ax.plot([r["beam_y_cm"] for r in radial_good],
            [r["r_u"][t] for r in radial_good], "o", color=tol_color[t],
            ms=4, label=f"measured, ±{int(t*100)}%")
ax.axhline(CLOUD_R_UM, color="k", ls="--", lw=1.2,
           label="Sr cloud radius (150 µm)")
ax.axhspan(0, 21, color="0.85", zorder=0, label="metric floor (≤21 µm)")
ax.set_yscale("log")
ax.set_ylim(10, 3000)
ax.set_xlim(0, 200)
ax.set_xlabel("distance after axicon3 (cm)")
ax.set_ylabel("uniform-intensity radius r$_u$ (µm)")
ax.set_title("Near-axis uniform-intensity radius vs y — measured vs QDHT "
             "model.\n" + OPTIC_TITLE, fontsize=10)
ax.legend(fontsize=8, loc="upper left", ncol=2)
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("uniform_radius_vs_y")
plt.show()

In [ ]:
# --- cloud-footprint contrast + example radial profiles ----------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(z_model_cm, model_c150, "-", color="tab:orange", lw=1.4,
         label="QDHT model")
gy = np.array([r["beam_y_cm"] for r in radial_good])
gc = np.array([r["C150"] for r in radial_good])
ax1.plot(gy, gc, "o", color="tab:green", ms=4, label="measured")
ax1.axhline(0.2, color="k", ls="--", lw=1, label="±10% uniformity (C=0.2)")
ax1.set_yscale("log")
ax1.set_xlim(0, 200)
ax1.set_xlabel("distance after axicon3 (cm)")
ax1.set_ylabel("contrast over r ≤ 150 µm")
ax1.set_title("Intensity contrast across the 300 µm cloud footprint",
              fontsize=9)
ax1.legend(fontsize=8)
ax1.minorticks_on()
ax1.grid(True, which="both", alpha=0.3)

for target in (50.0, 100.5, 148.0, 156.0):
    cands = [r for r in radial_good if abs(r["beam_y_cm"] - target) < 0.4]
    if not cands:
        continue
    r = max(cands, key=lambda r: r["I0_rate"])
    ax2.plot(r["r_um"][r["r_um"] <= 600],
             (r["mean"] / np.nanmax(r["mean"]))[r["r_um"] <= 600],
             lw=1.2, label=f"y = {r['beam_y_cm']:g} cm")
ax2.axvspan(0, CLOUD_R_UM, color="tab:blue", alpha=0.10)
ax2.text(CLOUD_R_UM / 2, 1.04, "cloud", ha="center", fontsize=8,
         color="tab:blue")
ax2.set_xlim(0, 600)
ax2.set_ylim(0, 1.1)
ax2.set_xlabel("radius about beam axis (µm)")
ax2.set_ylabel("azimuthal-mean intensity (each ÷ its own max)")
ax2.set_title("Measured radial profiles", fontsize=9)
ax2.legend(fontsize=8)
ax2.minorticks_on()
ax2.grid(True, alpha=0.3)
fig.suptitle("300 µm Sr-cloud uniformity check. " + OPTIC_TITLE, fontsize=9)
plt.tight_layout()
save_fig("cloud_uniformity_check")
plt.show()

win = (gy > 136) & (gy < 177)
print(f"window slices (136-177 cm): n={win.sum()}")
print(f"  measured C150 median {np.median(gc[win]):.2f} "
      f"(model {np.median(model_c150[(z_model_cm > 136) & (z_model_cm < 177)]):.2f}); "
      f"±10% uniformity needs C150 <= 0.2")
for t in UNIF_TOLS:
    rv = np.array([r["r_u"][t] for r in radial_good])
    print(f"  measured r_u(±{int(t*100)}%) in window: median "
          f"{np.median(rv[win]):.0f} µm (cloud needs 150 µm)")

### Verdict: the 300 µm cloud does NOT fit the uniformly bright region ⚠

Measured, across the fully-scanned window (y ≈ 136–177 cm, ~100 clean
slices):

- **r_u(±10%) ≤ 21 µm and r_u(±50%) ≈ 28–34 µm** — the bright region is
  uniform only over the inner ~quarter of the J₀ core. The cloud needs
  150 µm: a shortfall of **~×7 in radius even at ±50% tolerance**.
- **C150 ≈ 3.6** (model: 4.0): across the cloud footprint the intensity
  swings ~360% of its mean — the footprint spans the core, the first *null*,
  and about two full-contrast rings (visible directly in the measured
  profiles: FWHM ≈ 58 µm core, first zero ≈ 62 µm, rings at ~95/180/260 µm).
  ±10% uniformity corresponds to C150 ≤ 0.2, a factor of ~18 away.

This is not an alignment or quality problem — the measured profiles match the
ideal model closely; it is the geometry of a J₀ beam at k_r ≈ 3.9×10⁴ rad/m.
The uniform-core radius scales as 1/k_r: flat-to-±10% over 150 µm requires
k_r ≈ 3.0×10³ rad/m (effective α₃ ≈ 0.04°), and with the current annulus
radius that pushes the window center from 1.6 m out to **~20 m** — not a
bench-compatible knob. Options to discuss with the group:

1. **Relax the requirement**: if the cloud can live with the *core* rather
   than a flat plateau, the natural matched size is the core FWHM (~58 µm
   diameter) — a ~120 µm-diameter cloud region sees the central lobe only,
   with 100% peak-to-edge variation.
2. **Rescale the optic**: magnifying the output telescope by M multiplies
   the core by M and the window length by M²; reaching 150 µm at ±10% needs
   M ≈ 13 (window tens of meters — ruled out). A partial M (2–3×) plus
   tolerance relaxation may find a compromise.
3. **Different beam concept for uniformity**: flat-top shaping (or imaging
   an aperture) if transverse uniformity over 300 µm is a hard requirement —
   at the cost of the Bessel beam's long depth of focus, which was the point
   of this design.

Cross-check note: overlapping runs at the same y differ in absolute I₀ by up
to ~2× (independent exposure calibrations + laser drift between runs) —
r_u and C150 are per-slice normalized and unaffected.

## Beam cross-section map: measured vs model

The full r–y picture — each slice collapsed to its azimuthal-mean radial
profile (4-px ≈ 27.6 µm bins, r ≤ 8 mm, about the fitted/refined axis) and
stacked along y, next to the QDHT model on the same log range. One slice per
y (unsaturated preferred, then brightest I₀); columns break at coverage gaps.
Run-to-run absolute-brightness steps (~2×, independent exposure calibrations
+ laser drift) appear as vertical seams; structure within each column is
unaffected. Slices with known-contaminated centers are excluded by name with
a printed note.

In [ ]:
# --- measured r-y map --------------------------------------------------------
from matplotlib.colors import LogNorm

MAP_R_MM, MAP_BIN_PX = 8.0, 4
MAP_EXCLUDE = {
    # find-beam sweep frames left a false bright spot; both centering paths
    # mislocate the axis and smear the annulus
    "auto_scan-2026-07-23_09-48-16/y0026.00cm",
}

def wide_profile(slice_dir):
    """Azimuthal-mean profile to MAP_R_MM (coarse bins), counts/us."""
    comp = np.load(slice_dir / "composite.npy")
    meta = json.loads((slice_dir / "composite_meta.json").read_text())
    ext, s = meta["Extent_mm"], meta["mm_per_px"]
    thr = max(3.0, 0.25 * comp.max())
    rows, cols = np.nonzero(comp > thr)
    xc, zc, _ = taubin_circle_fit(ext["XMin"] + (cols + 0.5) * s,
                                  ext["ZMax"] - (rows + 0.5) * s,
                                  comp[rows, cols].astype(np.float64))
    col_c = (xc - ext["XMin"]) / s - 0.5
    row_c = (ext["ZMax"] - zc) / s - 0.5
    rr, cc = np.mgrid[0:comp.shape[0], 0:comp.shape[1]]
    near = (np.hypot(cc - col_c, rr - row_c) * s) <= 0.6
    if near.any() and comp[near].max() >= 0.5 * comp.max():
        pr, pc = np.unravel_index(np.argmax(np.where(near, comp, -1)), comp.shape)
        w = comp[max(pr-4, 0):pr+5, max(pc-4, 0):pc+5].astype(np.float64)
        yy, xx = np.mgrid[max(pr-4, 0):pr+5, max(pc-4, 0):pc+5]
        row_c = float((w * yy).sum() / w.sum())
        col_c = float((w * xx).sum() / w.sum())
    exposures, n_sat = set(), 0
    with (slice_dir / "frames.jsonl").open() as fh:
        for line in fh:
            if not line.strip():
                continue
            rec = json.loads(line)
            name = Path(rec["Path"]).name
            if "background" in name or "-dark" in name:
                continue
            exposures.add(round(rec["Extra"]["Exposure_us"], 1))
            n_sat += rec.get("SaturatedPixelCount", 0) > 0
    if len(exposures) != 1:
        raise ValueError(f"{len(exposures)} distinct exposures")
    T = exposures.pop()
    rbin = (np.hypot(cc - col_c, rr - row_c) / MAP_BIN_PX).astype(int)
    nbm = int(MAP_R_MM / (s * MAP_BIN_PX))
    ok = rbin < nbm
    sums = np.bincount(rbin[ok], weights=comp[ok].astype(np.float64), minlength=nbm)
    cnts = np.bincount(rbin[ok], minlength=nbm)
    expected = 2 * np.pi * (np.arange(nbm) + 0.5) * MAP_BIN_PX**2
    mean = np.where(cnts > 0.3 * expected, sums / np.maximum(cnts, 1), np.nan) / T
    return meta["BeamY_mm"] / 10.0, mean, s * MAP_BIN_PX, bool(n_sat > 0)


profiles = []
for run_dir in sorted(GANTRY_ROOT.glob("auto_scan-*")):
    for slice_dir in sorted(run_dir.glob("y*cm")):
        if not (slice_dir / "composite_meta.json").exists():
            continue
        rel = str(slice_dir.relative_to(GANTRY_ROOT))
        if rel in MAP_EXCLUDE:
            print(f"excluding {rel} (known-contaminated center)")
            continue
        try:
            y_cm, mean, rstep, sat = wide_profile(slice_dir)
        except (ValueError, OSError, np.linalg.LinAlgError) as e:
            print(f"skipping {rel}: {type(e).__name__}: {e}")
            continue
        profiles.append((y_cm, mean, rstep, sat))

by_y = {}
for y_cm, mean, rstep, sat in profiles:
    key = round(y_cm, 2)
    I0 = np.nanmean(mean[:2])
    cur = by_y.get(key)
    if cur is None or (cur[2] and not sat) or (cur[2] == sat and I0 > cur[1]):
        by_y[key] = (mean, I0, sat, rstep)
ys = np.array(sorted(by_y))
rstep = by_y[ys[0]][3]
nbm = len(by_y[ys[0]][0])
Mmap = np.full((nbm, len(ys)), np.nan)
for j, y in enumerate(ys):
    Mmap[:, j] = by_y[y][0]
print(f"{len(ys)} y-columns, r to {nbm * rstep:.1f} mm")

In [ ]:
# --- map figure: measured vs model, same log range ---------------------------
edges = np.empty(len(ys) + 1)
edges[1:-1] = 0.5 * (ys[:-1] + ys[1:])
edges[0], edges[-1] = ys[0] - 0.5, ys[-1] + 0.5
gap = np.where(np.diff(ys) > 2.5)[0]
r_edges = np.arange(nbm + 1) * rstep

r_map_m = np.linspace(0.0, nbm * rstep * 1e-3, 480)
M_map = model["q"].eval_matrix(r_map_m)
Imod = np.empty((len(r_map_m), len(z_model_cm)))
for i0 in range(0, len(z_model_cm), 64):
    Imod[:, i0:i0 + 64] = (np.abs(model["Y3"][i0:i0 + 64] @ M_map) ** 2).T

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12.5, 8), sharex=True)
vmax = np.nanmax(Mmap)
cmap = plt.get_cmap("inferno").copy()
cmap.set_bad("0.25")
Mmk = np.ma.masked_invalid(Mmap)
for j0, j1 in zip(np.r_[0, gap + 1], np.r_[gap + 1, len(ys)]):
    pm = ax1.pcolormesh(edges[j0:j1 + 1], r_edges, Mmk[:, j0:j1],
                        norm=LogNorm(vmin=vmax * 1e-4, vmax=vmax, clip=True),
                        cmap=cmap, shading="flat")
ax1.set_ylabel("r about beam axis (mm)")
ax1.set_title("Measured (one slice per y)", fontsize=10)
fig.colorbar(pm, ax=ax1, label="counts/µs (log)")

pm2 = ax2.pcolormesh(z_model_cm, r_map_m * 1e3, Imod,
                     norm=LogNorm(vmin=Imod.max() * 1e-4, vmax=Imod.max(),
                                  clip=True),
                     cmap="inferno", shading="auto")
ax2.set_xlim(0, 200)
ax2.set_ylabel("r (mm)")
ax2.set_xlabel("distance after axicon3 (cm)")
ax2.set_title("QDHT model (same log range)", fontsize=10)
fig.colorbar(pm2, ax=ax2, label="model intensity (arb., log)")

fig.suptitle("Beam cross-section vs distance: converging annulus → Bessel "
             "window. Gray = no gantry coverage.\n" + OPTIC_TITLE, fontsize=10)
plt.tight_layout()
save_fig("rz_map_measured_vs_model")
plt.show()

## What-if: input beam diameter ×2, all else constant

Doubling the input 1/e² diameter (4.59 → 9.18 mm) with the same axicons and
separations. The key scale is zmax = w₀/θ (the distance over which the two
cones from axicon 1 overlap): it doubles from 57 to 114 mm, so L₁₂ = 190 mm
drops from 3.3×zmax (annulus cleanly separated at axicon 2) to 1.7×zmax — the
annulus at axicon 3 is **no longer cleanly formed**, and light interior to it
contaminates the axis. Both cases are re-propagated on a common z grid (the
auto range from the ring-width estimate is unusable for the 2× case, for this
same reason) and compared at **equal input power** (fixed peak amplitude
would quadruple the power with 2× diameter).

Numerical caveat: at 2× diameter the fixed N = 4096 grid resolves the axicon
fringes at 3.5 pts/fringe and kr_max/kr = 1.7 (vs 5.0 and 2.5 at baseline) —
above the module's guards, but treat fine spike detail as marginally
resolved.

In [ ]:
# --- simulate D x2 and compare on a common z grid ----------------------------
from simulator.qdht_axicon import K0, propagate_spectrum

OPTIC_2X = dict(OPTIC, GaussianBeamWaist_mm=2 * OPTIC["GaussianBeamWaist_mm"])
pair_2x, model_2x = simulate_recorded_optic(OPTIC_2X, N=4096, Nz2=40, Nz3=50)

z_common = np.linspace(1e-3, 2.8, 500)
onax_c = {}
for tag, mdl in (("1x", model), ("2x", model_2x)):
    Y = propagate_spectrum(mdl["y3"], mdl["q"].kr, K0, z_common)
    onax_c[tag] = (mdl["q"].onax(Y), Y)
pw = pair["P_in"] / pair_2x["P_in"]     # equal-input-power rescale for 2x
z_c_cm = z_common * 100


def window_stats(z, y):
    i = int(np.argmax(y))
    half = 0.5 * y[i]
    lo = np.where(y[:i] < half)[0]
    hi = np.where(y[i:] < half)[0]
    return z[i], (z[i + hi[0]] if len(hi) else z[-1]) - (z[lo[-1]] if len(lo) else z[0])


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(z_c_cm, onax_c["1x"][0], "-", color="tab:orange", lw=1.5,
         label="D = 4.59 mm (as built)")
ax1.plot(z_c_cm, onax_c["2x"][0] * pw, "-", color="tab:cyan", lw=1.5,
         label="D = 9.18 mm (2×), equal input power")
ax1.set_xlim(0, 280)
ax1.set_xlabel("distance after axicon3 (cm)")
ax1.set_ylabel("on-axis intensity (arb., common scale)")
ax1.set_title("Axial window", fontsize=9)
ax1.legend(fontsize=8)
ax1.minorticks_on()
ax1.grid(True, alpha=0.3)

r_eval = np.linspace(0, 300e-6, 1000)
for (tag, (on, Y)), w, c, lab in zip(onax_c.items(), (1.0, pw),
                                     ("tab:orange", "tab:cyan"),
                                     ("D = 4.59 mm", "D = 9.18 mm (2×)")):
    mdl = model if tag == "1x" else model_2x
    iz = int(np.argmax(on))
    row = np.abs(Y[iz] @ mdl["q"].eval_matrix(r_eval)) ** 2
    ax2.plot(r_eval * 1e6, row / row[0], color=c, lw=1.4, label=lab)
ax2.axvspan(0, 150, color="tab:blue", alpha=0.10)
ax2.text(75, 1.04, "cloud", ha="center", fontsize=8, color="tab:blue")
ax2.set_xlabel("radius (µm)")
ax2.set_ylabel("intensity ÷ on-axis (at each window peak)")
ax2.set_title("Radial profile at the window peak", fontsize=9)
ax2.legend(fontsize=8)
ax2.minorticks_on()
ax2.grid(True, alpha=0.3)
fig.suptitle("Input beam diameter ×2, all else constant (QDHT model). "
             + OPTIC_TITLE, fontsize=9)
plt.tight_layout()
save_fig("input_diameter_2x")
plt.show()

for tag, w, pr in (("1x", 1.0, pair), ("2x", pw, pair_2x)):
    on = onax_c[tag][0] * w
    zp, fw = window_stats(z_c_cm, on)
    print(f"{tag}: zmax = {pr['zmax']*1e3:5.1f} mm (L12/zmax = "
          f"{0.190/pr['zmax']:.1f})  window peak z = {zp:6.1f} cm  "
          f"FWHM = {fw:5.1f} cm  peak on-axis = {on.max():.0f}")

### Reading the ×2 result

- **The window degrades, it doesn't just move.** The clean 36 cm-FWHM window
  at 163 cm becomes a long rippled pedestal (~40–190 cm) with sharp
  interference spikes — the signature of on-axis contamination from the
  unseparated annulus. Peak on-axis intensity drops ~2× *at equal input
  power*, despite the naive expectation that a longer zmax builds a longer,
  stronger window.
- **The near-axis structure gets worse, not better**: the radial profile at
  the (spiky) peak is modulated inside the core, and C150 rises from ~4.2 to
  ~6.7. Doubling D does nothing for the 300 µm-cloud uniformity problem —
  the core scale is set by k_r₃ alone.
- **To use a 2× beam properly, L₁₂ must scale with it**: the clean-annulus
  condition L₁₂ ≳ 3×zmax needs L₁₂ ≳ 340 mm. With that change, the 2× beam
  would double the ring width Δ ≈ w₀ and hence the *window length*, at
  correspondingly lower peak intensity — the legitimate way to trade input
  size for depth of focus.
- This also retroactively supports the jul13 waist-reading check: with the
  "radius" interpretation (D = 9.18 mm) the model is in this contaminated
  regime and disagrees with the data everywhere, while the "diameter" reading
  matches — consistent with the clean measured windows above.

## L₁₂ sweep at doubled input diameter

Follow-up to the ×2 what-if: how much L₁₂ does the 9.18 mm beam need, and
what does the recovered window look like? Sweep L₁₂ = 190–450 mm (1.7–3.9 ×
the doubled zmax = 114 mm), all else fixed, equal input power, with the
as-built system as reference.

Numerics: the larger annulus radii at long L₁₂ need a finer radial grid —
this cell uses **N = 8192** (T-matrix ~0.5 GB, built once and cached across
the sweep; expect a few minutes). Resolution checks at two N values confirm
the clean-case windows are grid-converged; the narrow ~50 cm-periodic spikes
belong to the broken L₁₂ = 190 mm case only (revivals of the unseparated
annulus light), not to numerics.

In [ ]:
# --- L12 sweep at D = 9.18 mm, equal input power -----------------------------
from simulator.qdht_axicon import mm as MM, simulate_axicon_pair, simulate_third_axicon

D_2X = 9.18 * MM
L12_SWEEP_MM = [190, 230, 270, 310, 350, 400, 450]
N_SWEEP = 8192
z_sw = np.linspace(1e-3, 5.0, 600)
z_sw_cm = z_sw * 100

# as-built reference in the same equal-power units
pair_ref = simulate_axicon_pair(D=4.59 * MM, alpha_deg=OPTIC["Axicon1_deg"],
                                L_sep=OPTIC["L12_mm"] * MM,
                                z_after=2 * OPTIC["L23_mm"] * MM,
                                N=N_SWEEP, Nz1=4, Nz2=8,
                                pad=2.5 * 4.59 * MM / 2, verbose=False)
model_ref = simulate_third_axicon(pair_ref, OPTIC["Axicon3_deg"],
                                  L23=OPTIC["L23_mm"] * MM, Nz3=8,
                                  verbose=False)
onax_ref = model_ref["q"].onax(propagate_spectrum(
    model_ref["y3"], model_ref["q"].kr, K0, z_sw)) / pair_ref["P_in"]

sweep = []
for L12 in L12_SWEEP_MM:
    pr = simulate_axicon_pair(D=D_2X, alpha_deg=OPTIC["Axicon1_deg"],
                              L_sep=L12 * MM,
                              z_after=2 * OPTIC["L23_mm"] * MM,
                              N=N_SWEEP, Nz1=4, Nz2=8,
                              pad=2.5 * D_2X / 2, verbose=False)
    mdl = simulate_third_axicon(pr, OPTIC["Axicon3_deg"],
                                L23=OPTIC["L23_mm"] * MM, Nz3=8,
                                verbose=False)
    Y = propagate_spectrum(mdl["y3"], mdl["q"].kr, K0, z_sw)
    on = mdl["q"].onax(Y) / pr["P_in"]
    i_pk = int(np.argmax(on))
    half = 0.5 * on[i_pk]
    lo = np.where(on[:i_pk] < half)[0]
    hi = np.where(on[i_pk:] < half)[0]
    fwhm = (z_sw_cm[i_pk + hi[0]] if len(hi) else z_sw_cm[-1]) - \
           (z_sw_cm[lo[-1]] if len(lo) else z_sw_cm[0])
    r_eval = np.linspace(0, 150e-6, 300)
    row = np.abs(Y[i_pk] @ mdl["q"].eval_matrix(r_eval)) ** 2
    span = (z_sw_cm > z_sw_cm[i_pk] - fwhm / 2) & (z_sw_cm < z_sw_cm[i_pk] + fwhm / 2)
    sweep.append(dict(L12=L12, ratio=L12 * MM / pr["zmax"], onax=on,
                      z_pk=z_sw_cm[i_pk], fwhm=fwhm, peak=on[i_pk],
                      c150=(row.max() - row.min()) / row.mean(),
                      ripple=on[span].std() / on[span].mean()))
    print(f"L12={L12:3d} mm ({sweep[-1]['ratio']:.2f} zmax): "
          f"peak z={sweep[-1]['z_pk']:6.1f} cm  FWHM={fwhm:5.1f} cm  "
          f"ripple={sweep[-1]['ripple']:.2f}  C150={sweep[-1]['c150']:.2f}")

In [ ]:
# --- sweep figure ------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.2),
                               constrained_layout=True)
cmap_sw = plt.get_cmap("viridis")
ax1.plot(z_sw_cm, onax_ref, "--", color="tab:orange", lw=1.4,
         label="as built: D = 4.59 mm, L12 = 190 mm")
for i, r in enumerate(sweep):
    ax1.plot(z_sw_cm, r["onax"], lw=1.2,
             color=cmap_sw(0.1 + 0.8 * i / (len(sweep) - 1)),
             label=f"L12 = {r['L12']} mm ({r['ratio']:.1f} zmax)")
ax1.set_xlim(0, 500)
ax1.set_xlabel("distance after axicon3 (cm)")
ax1.set_ylabel("on-axis intensity / input power (arb.)")
ax1.set_title(f"D = {D_2X/MM:g} mm: axial window vs L12", fontsize=9)
ax1.legend(fontsize=7)
ax1.minorticks_on()
ax1.grid(True, alpha=0.3)

L = [r["L12"] for r in sweep]
ax2.plot(L, [r["peak"] for r in sweep], "o-", color="tab:orange",
         label="peak on-axis (arb.)")
ax2b = ax2.twinx()
ax2b.plot(L, [r["ripple"] for r in sweep], "s-", color="tab:red",
          label="in-window ripple (std/mean)")
ax2b.plot(L, [r["fwhm"] / 100 for r in sweep], "^-", color="tab:blue",
          label="window FWHM (m)")
ax2.axvline(3 * 114.4, color="0.5", ls="--", lw=1)   # 3 x zmax(D=9.18)
ax2.annotate("3 zmax", (3 * 114.4 + 4, 0.02),
             xycoords=("data", "axes fraction"), fontsize=8, color="0.4")
ax2.set_xlabel("L12 (mm)")
ax2.set_ylabel("peak on-axis (arb.)", color="tab:orange")
ax2b.set_ylabel("ripple / FWHM (m)")
h1, l1 = ax2.get_legend_handles_labels()
h2, l2 = ax2b.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, fontsize=8, loc="center right")
ax2.set_title("Window metrics vs L12", fontsize=9)
ax2.minorticks_on()
ax2.grid(True, alpha=0.3)
fig.suptitle("L12 sweep at doubled input diameter (QDHT model, equal input "
             "power). " + OPTIC_TITLE.replace("\n", " "), fontsize=9)
save_fig("l12_sweep_d2x")
plt.show()

### Reading the L₁₂ sweep

- **Recovery is fast**: by L₁₂ ≈ 230 mm (2.0 zmax) the window is already
  mostly clean, and from ~270 mm (2.4 zmax) the near-axis structure is fully
  restored — C150 locks at 4.18, identical to the as-built system. The
  "≳3 zmax" rule of thumb is conservative; ~2.5 zmax suffices at this
  geometry.
- **The trade is exactly the conservation argument**: every clean 2×-diameter
  window is ~85 cm FWHM (2.3× the as-built 36 cm) at ~2.2× lower peak
  on-axis intensity, equal input power. Doubling the annulus width Δ ≈ w₀
  stretches the same light over a proportionally longer window.
- **Window position scales with L₁₂** (peak ≈ 1.6 → 4.1 m across the sweep):
  choosing L₁₂ is choosing where the window sits. A 2× beam with
  L₁₂ ≈ 270–310 mm puts a clean ~85 cm window at ~2.2–2.5 m — bench-feasible
  if that standoff is wanted.
- **Still no help for the 300 µm cloud**: C150 stays ~4.2 throughout — the
  core scale is k_r₃'s alone, as established in the uniformity section. This
  knob buys depth of focus and standoff, not transverse uniformity.
- The elevated ripple metric at L₁₂ = 400–450 mm reflects the double-humped
  window tops (visible in the curves), not spikes — the broken L₁₂ = 190 mm
  case is the only one with sharp revival spikes.